In [6]:
import os
from dotenv import load_dotenv
from pymongo import MongoClient, UpdateOne

load_dotenv()

MONGO_URI = os.getenv("MONGODB_URI")
if not MONGO_URI:
    raise RuntimeError("MONGO_URI not found in .env")
client = MongoClient(MONGO_URI)
db = client["hr-cleaned"]

In [7]:
DB_NAME = "hr-cleaned"
COLLECTION_NAME = "base_report"

client = MongoClient(MONGO_URI)
collection = client[DB_NAME][COLLECTION_NAME]


def build_manager_details(manager_doc):
    if manager_doc is None:
        return {
            "manager name": "NA",
            "manager mail": "NA",
            "manager designation": "NA"
        }

    return {
        "manager name": f"{manager_doc.get('first name', 'NA')} {manager_doc.get('last name', 'NA')}",
        "manager mail": manager_doc.get("email", "NA"),
        "manager designation": manager_doc.get("designation", "NA")
    }


def enrich_with_manager_details():
    docs = list(collection.find({}, {
        "_id": 1,
        "employee code": 1,
        "manager employee code": 1,
        "first name": 1,
        "last name": 1,
        "email": 1,
        "designation": 1
    }))

    lookup = {doc.get("employee code"): doc for doc in docs}

    bulk_ops = []

    for doc in docs:
        manager_emp_code = doc.get("manager employee code")

        manager_doc = lookup.get(manager_emp_code) if manager_emp_code else None
        manager_details = build_manager_details(manager_doc)

        bulk_ops.append(
            UpdateOne(
                {"_id": doc["_id"]},
                {"$set": {"manager details": manager_details}}
            )
        )

    if bulk_ops:
        result = collection.bulk_write(bulk_ops)
        print("Matched:", result.matched_count)
        print("Modified:", result.modified_count)
    else:
        print("No documents to update.")

In [8]:
enrich_with_manager_details()

Matched: 7859
Modified: 7859
